# Phase 1 — Re-scoring sémantique SONAR des paires minées

Ce kernel recalcule, pour chaque paire minée candidate (sortie de la **Phase 0**),
une similarité sémantique **cosinus** via les embeddings **SONAR** (200 langues NLLB, dont `ewe_Latn`),
implémenté en **pur transformers** (port `cointegrated/SONAR_200_text_encoder`, **sans fairseq2**).
But : distinguer les vraies traductions des **désalignements sémantiques** que le langid ne détecte pas.

**Pré-requis (UNE fois, dans l'UI Kaggle) :**
1. *Add-ons ▸ Secrets* ▸ ajouter un secret `HF_TOKEN` = **jeton Hugging Face WRITE**
   (les datasets `ewe-mined-candidates` et `ewe-en-fr-nllb-translation` sont privés et on repousse les résultats).
2. *Settings* ▸ **Accelerator = GPU T4 x2**, **Environment = « Always use latest environment »**, **Internet ON**.
   (Un environnement épinglé/ancien provoque l'erreur CUDA « no kernel image ».)

**Optimisations GPU** : inférence FP16, gros batch, tri par longueur, dédoublonnage intra-tranche,
checkpoints tous les 500 000 lignes repoussés sur le Hub (reprise si la session coupe).

> Repli **LASER3** (encodeur alternatif couvrant l'éwé) : voir la dernière cellule.

In [ ]:
# 1) Dependances : SONAR via transformers (port pur PyTorch, SANS fairseq2)
# On s'appuie sur transformers/torch deja presents sur Kaggle (compatibilite garantie).
# Seul sentencepiece (tokenizer NLLB-200) peut manquer.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=False)
import transformers, torch, huggingface_hub
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.version.cuda, "| huggingface_hub", huggingface_hub.__version__)

# Diagnostic GPU fail-fast : le torch de Kaggle doit supporter l'architecture du GPU assigne.
if not torch.cuda.is_available():
    raise SystemExit("Aucun GPU detecte -> Settings > Accelerator = GPU T4 x2.")
cap = torch.cuda.get_device_capability(0)
sm = f"sm_{cap[0]}{cap[1]}"
archs = torch.cuda.get_arch_list()
print("GPU:", torch.cuda.get_device_name(0), "| capability", sm, "| torch archs:", archs)
if sm not in archs:
    raise SystemExit(
        f"GPU {sm} non supporte par ce torch (archs={archs}) -> erreur 'no kernel image'.\n"
        "CORRECTIF (UI Kaggle): Settings > Environment > 'Always use latest environment' ; "
        "Accelerator = GPU T4 x2 ; puis relancer le kernel.")
print("Installation terminee. GPU compatible.")

In [ ]:
# 2) Vérification rapide (fail-fast) des imports
try:
    from transformers import AutoTokenizer  # noqa: F401
    from transformers.models.m2m_100.modeling_m2m_100 import M2M100Encoder  # noqa: F401
    print("OK: transformers + M2M100Encoder disponibles (SONAR sans fairseq2).")
except Exception as e:
    raise SystemExit(
        "ECHEC import transformers/M2M100Encoder : " + repr(e) +
        "\n-> Verifier la version de transformers sur Kaggle."
        "\n-> Ou utiliser le repli LASER3 (derniere cellule).")

In [ ]:
# 3) Configuration + authentification Hugging Face (secret Kaggle)
import os, torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)

CANDIDATES_REPO = "romaricnadjire/ewe-mined-candidates"   # candidats Phase 0 + sorties Phase 1
CLEAN_REPO      = "romaricnadjire/ewe-en-fr-nllb-translation"  # 307k propres (calibration)
SCORES_SUBDIR   = "sonar_scores"   # checkpoints (idx, cos)
OUT_SUBDIR      = "realigned"      # datasets filtres finaux
SONAR_MODEL     = "cointegrated/SONAR_200_text_encoder"  # SONAR en pur transformers (sans fairseq2)

CHUNK      = 500_000   # taille des tranches (checkpoint)
BATCH      = 64        # batch SONAR (OOM -> 32 ; marge -> 128)
CALIB_PCTL = 5         # percentile (p5) de la distribution propre -> seuil

FILES = [
    # (prefixe, fichier, langue_ewe, langue_cible) -- fichiers gzip (upload plus fiable)
    ("ee_en", "ewe_en.jsonl.gz", "ewe_Latn", "eng_Latn"),
    ("ee_fr", "ewe_fr.jsonl.gz", "ewe_Latn", "fra_Latn"),
]

DEVICE = torch.device("cuda")
WORK = "/kaggle/working"
os.makedirs(f"{WORK}/{SCORES_SUBDIR}", exist_ok=True)
os.makedirs(f"{WORK}/{OUT_SUBDIR}", exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 4) Telechargement des candidats (Phase 0) + jeu propre (calibration)
import os
from huggingface_hub import hf_hub_download

cand_paths = {}
for prefix, fname, _, _ in FILES:
    p = hf_hub_download(CANDIDATES_REPO, fname, repo_type="dataset",
                        local_dir=f"{WORK}/candidates")
    cand_paths[prefix] = p
    print("candidat:", prefix, "->", p, f"({os.path.getsize(p)/1e6:.1f} Mo)")

clean_train = hf_hub_download(CLEAN_REPO, "train.jsonl", repo_type="dataset",
                              local_dir=f"{WORK}/clean")
print("propre:", clean_train, f"({os.path.getsize(clean_train)/1e6:.1f} Mo)")

In [ ]:
# 5) Chargement de l'encodeur SONAR (port transformers, FP16, sur GPU)
from transformers import AutoTokenizer
from transformers.models.m2m_100.modeling_m2m_100 import M2M100Encoder

tokenizer = AutoTokenizer.from_pretrained(SONAR_MODEL)
encoder = (M2M100Encoder.from_pretrained(SONAR_MODEL, torch_dtype=torch.float16)
           .to(DEVICE).eval())
print("Encodeur SONAR pret (fp16) sur", DEVICE, "| dim =", encoder.config.d_model)

In [ ]:
# 6) Fonctions utilitaires (embeddings normalises, cosinus par paire, lecture jsonl)
import json, gzip, numpy as np, torch

EMB_DIM = encoder.config.d_model   # 1024 (SONAR)
MAXLEN  = 256

@torch.inference_mode()
def _encode(texts, lang):
    # Encode un batch -> embedding moyen masque, normalise (cpu fp16).
    tokenizer.src_lang = lang
    batch = tokenizer(texts, return_tensors="pt", padding=True,
                      truncation=True, max_length=MAXLEN).to(DEVICE)
    hid = encoder(**batch).last_hidden_state            # (B, T, EMB_DIM)
    mask = batch.attention_mask.unsqueeze(-1)           # (B, T, 1)
    emb = (hid * mask).sum(1) / mask.sum(1).clamp(min=1)
    return torch.nn.functional.normalize(emb.float(), dim=1).half().cpu()

@torch.inference_mode()
def embed(texts, lang, bs=BATCH):
    # Dedoublonnage + tri par longueur pour economiser le GPU.
    uniq = list(dict.fromkeys(texts))
    pos = {t: i for i, t in enumerate(uniq)}
    order = sorted(range(len(uniq)), key=lambda i: len(uniq[i]))
    out_uniq = torch.empty((len(uniq), EMB_DIM), dtype=torch.float16)
    for s in range(0, len(order), bs):
        idxs = order[s:s + bs]
        vecs = _encode([uniq[i] for i in idxs], lang)
        for k, oi in enumerate(idxs):
            out_uniq[oi] = vecs[k]
    sel = torch.tensor([pos[t] for t in texts])
    return out_uniq[sel]

@torch.inference_mode()
def cosine_pairs(src, tgt, src_lang, tgt_lang, bs=BATCH):
    es = embed(src, src_lang, bs)
    et = embed(tgt, tgt_lang, bs)
    return (es.float() * et.float()).sum(dim=1).numpy()   # cos (vecteurs normalises)

def read_jsonl(path):
    op = gzip.open if str(path).endswith(".gz") else open
    with op(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

In [ ]:
# 7) Calibration du seuil sur les paires PROPRES (verite terrain)
import numpy as np, json

clean = {"ee_en": ([], []), "ee_fr": ([], [])}
for row in read_jsonl(clean_train):
    tr = row.get("translation", row)
    ewe = tr.get("ewe_Latn")
    if not ewe:
        continue
    if tr.get("eng_Latn"):
        clean["ee_en"][0].append(ewe); clean["ee_en"][1].append(tr["eng_Latn"])
    elif tr.get("fra_Latn"):
        clean["ee_fr"][0].append(ewe); clean["ee_fr"][1].append(tr["fra_Latn"])

THRESHOLDS = {}
for prefix, _, ewe_lang, tgt_lang in FILES:
    src, tgt = clean[prefix]
    if not src:
        continue
    cos = cosine_pairs(src, tgt, ewe_lang, tgt_lang)
    thr = float(np.percentile(cos, CALIB_PCTL))
    THRESHOLDS[prefix] = thr
    print(f"[{prefix}] propre n={len(src)} | moy={cos.mean():.3f} med={np.median(cos):.3f} "
          f"p5={np.percentile(cos,5):.3f} p10={np.percentile(cos,10):.3f} -> seuil={thr:.3f}")

with open(f"{WORK}/{OUT_SUBDIR}/thresholds.json", "w") as f:
    json.dump(THRESHOLDS, f, indent=2)
print("Seuils:", THRESHOLDS)

In [ ]:
# 8) Scoring des candidats par tranches de 500k + checkpoints repousses sur le Hub
import os, json, numpy as np
from huggingface_hub import HfApi, list_repo_files

api = HfApi()

def done_offsets(prefix):
    try:
        files = list_repo_files(CANDIDATES_REPO, repo_type="dataset")
    except Exception:
        files = []
    done = set()
    for f in files:
        base = os.path.basename(f)
        if f.startswith(f"{SCORES_SUBDIR}/scores_{prefix}_") and base.endswith(".jsonl"):
            try:
                done.add(int(base.split("_")[-1].split(".")[0]))
            except ValueError:
                pass
    return done

def flush(prefix, ewe_lang, tgt_lang, off, src, tgt, gidx, done):
    if not src:
        return
    if off in done:
        print(f"  tranche {off}: deja sur le Hub, ignoree.", flush=True)
        return
    cos = cosine_pairs(src, tgt, ewe_lang, tgt_lang)
    out = f"{WORK}/{SCORES_SUBDIR}/scores_{prefix}_{off:09d}.jsonl"
    with open(out, "w", encoding="utf-8") as f:
        for gi, c in zip(gidx, cos):
            f.write(json.dumps({"idx": gi, "cos": float(c)}) + "\n")
    api.upload_file(path_or_fileobj=out,
                    path_in_repo=f"{SCORES_SUBDIR}/{os.path.basename(out)}",
                    repo_id=CANDIDATES_REPO, repo_type="dataset",
                    commit_message=f"SONAR scores {prefix} @ {off}")
    print(f"  tranche {off}: {len(src)} paires (cos moy={cos.mean():.3f}) -> Hub.", flush=True)

for prefix, fname, ewe_lang, tgt_lang in FILES:
    path = cand_paths[prefix]
    done = done_offsets(prefix)
    print(f"\n=== {prefix} ({fname}) | tranches deja faites: {sorted(done)} ===", flush=True)
    buf_src, buf_tgt, buf_idx, offset = [], [], [], 0
    for i, row in enumerate(read_jsonl(path)):
        tr = row.get("translation", row)
        buf_src.append(tr.get(ewe_lang, "")); buf_tgt.append(tr.get(tgt_lang, "")); buf_idx.append(i)
        if len(buf_src) >= CHUNK:
            flush(prefix, ewe_lang, tgt_lang, offset, buf_src, buf_tgt, buf_idx, done)
            offset = i + 1
            buf_src, buf_tgt, buf_idx = [], [], []
    flush(prefix, ewe_lang, tgt_lang, offset, buf_src, buf_tgt, buf_idx, done)
    print(f"=== {prefix} termine ===", flush=True)

In [ ]:
# 9) Fusion des scores + filtrage par seuil + statistiques
import os, json, numpy as np
from huggingface_hub import hf_hub_download, list_repo_files

api_files = list_repo_files(CANDIDATES_REPO, repo_type="dataset")
with open(f"{WORK}/{OUT_SUBDIR}/thresholds.json") as f:
    THRESHOLDS = json.load(f)

summary = {}
for prefix, fname, ewe_lang, tgt_lang in FILES:
    score_files = [f for f in api_files if f.startswith(f"{SCORES_SUBDIR}/scores_{prefix}_")]
    idx2cos = {}
    for sf in sorted(score_files):
        p = hf_hub_download(CANDIDATES_REPO, sf, repo_type="dataset", local_dir=f"{WORK}/scores_dl")
        for line in open(p, encoding="utf-8"):
            d = json.loads(line)
            idx2cos[d["idx"]] = d["cos"]
    thr = THRESHOLDS[prefix]
    kept = total = 0
    cos_all = []
    out_name = fname[:-3] if fname.endswith(".gz") else fname
    out_path = f"{WORK}/{OUT_SUBDIR}/{out_name}"
    with open(out_path, "w", encoding="utf-8") as out:
        for i, row in enumerate(read_jsonl(cand_paths[prefix])):
            c = idx2cos.get(i)
            if c is None:
                continue
            cos_all.append(c); total += 1
            if c >= thr:
                row["sonar_cos"] = c
                out.write(json.dumps(row, ensure_ascii=False) + "\n")
                kept += 1
    cos_all = np.array(cos_all)
    summary[prefix] = {
        "total_scored": total, "kept": kept,
        "kept_ratio": round(kept / max(total, 1), 4), "threshold": thr,
        "cos_mean": float(cos_all.mean()) if total else None,
        "cos_p50": float(np.percentile(cos_all, 50)) if total else None,
    }
    print(prefix, summary[prefix], flush=True)

with open(f"{WORK}/{OUT_SUBDIR}/phase1_report.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2))

In [ ]:
# 10) Push des datasets realignes + rapport vers le Hub
from huggingface_hub import HfApi
HfApi().upload_folder(
    folder_path=f"{WORK}/{OUT_SUBDIR}",
    repo_id=CANDIDATES_REPO, repo_type="dataset", path_in_repo=OUT_SUBDIR,
    allow_patterns=["*.jsonl", "*.json"],
    commit_message="Phase 1 SONAR: paires realignees (cos>=seuil) + rapport",
)
print("Resultats realignes pousses sur", CANDIDATES_REPO, "/", OUT_SUBDIR)

## Repli LASER3 (si l'installation SONAR/fairseq2 échoue)

LASER3 couvre aussi `ewe_Latn` et s'installe **sans fairseq2** :

```python
!pip install -q laser_encoders
from laser_encoders import LaserEncoderPipeline
import numpy as np

enc_ewe = LaserEncoderPipeline(lang="ewe_Latn")
enc_en  = LaserEncoderPipeline(lang="eng_Latn")   # ou lang="fra_Latn"

def cosine_pairs_laser(src, tgt, enc_s, enc_t):
    a = enc_s.encode_sentences(src); b = enc_t.encode_sentences(tgt)
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return (a * b).sum(1)
```

Il suffit alors de remplacer `cosine_pairs` par `cosine_pairs_laser` (calibration, checkpoints et
filtrage restent identiques).